# 01 — Data Audit

Audit of candidate training datasets for the NutriScan v1 classifier.

**Questions this notebook answers:**
1. What classes does each source provide, and how many images per class?
2. How bad is the class imbalance?
3. Which classes overlap between sources (merge candidates)?
4. Are the images usable (resolution, corrupt files)?
5. Does the draft `ml/labels.yaml` have enough coverage to be frozen?

Datasets are expected under `ml/data/raw/` (git-ignored), one folder per source,
each containing class subfolders of images (extracted, not archives).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yaml
from PIL import Image

sns.set_theme(style="whitegrid")

ML_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path("ml")
RAW = ML_DIR / "data" / "raw"
LABELS_FILE = ML_DIR / "labels.yaml"
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

draft = yaml.safe_load(LABELS_FILE.read_text())
draft_labels = set(draft["classes"])
print(f"Draft label set: {len(draft_labels)} classes (status: {draft['status']})")
print(f"Raw data dir: {RAW.resolve()}")

## 1. Discover sources and count images per class

In [ ]:
def normalize(name: str) -> str:
    """Normalize a class-folder name for cross-source comparison."""
    return name.strip().lower().replace(" ", "_").replace("-", "_")


def find_class_dirs(source_dir: Path) -> dict[str, Path]:
    """Find the deepest level that looks like class folders (dirs containing images)."""
    candidates: dict[str, Path] = {}
    for d in sorted(source_dir.rglob("*")):
        if not d.is_dir():
            continue
        has_images = any(f.suffix.lower() in IMG_EXTS for f in d.iterdir() if f.is_file())
        if has_images:
            candidates.setdefault(normalize(d.name), d)
    return candidates


rows = []
sources = [d for d in RAW.iterdir() if d.is_dir()] if RAW.exists() else []
print(f"Sources found: {[s.name for s in sources]}")
for src in sources:
    for cls, cdir in find_class_dirs(src).items():
        n = sum(1 for f in cdir.iterdir() if f.suffix.lower() in IMG_EXTS)
        rows.append({"source": src.name, "class": cls, "n_images": n, "path": str(cdir)})

df = pd.DataFrame(rows)
if df.empty:
    raise SystemExit("No datasets found under ml/data/raw — extract the downloads first.")
print(
    f"{df['source'].nunique()} sources, {df['class'].nunique()} distinct classes, "
    f"{df['n_images'].sum():,} images total"
)
df.groupby("source").agg(classes=("class", "nunique"), images=("n_images", "sum"))

## 2. Class imbalance

In [ ]:
per_class = df.groupby("class")["n_images"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 4))
per_class.plot.bar(ax=ax, width=0.9)
ax.set_yscale("log")
ax.set_ylabel("images (log)")
ax.set_title(f"Images per class across all sources (n={len(per_class)})")
ax.tick_params(axis="x", labelsize=5)
plt.tight_layout()
plt.show()

print(f"median {per_class.median():.0f} · min {per_class.min()} · max {per_class.max()}")
print(f"classes with <150 images: {(per_class < 150).sum()}")

## 3. Overlap between sources

In [ ]:
overlap = (
    df.groupby("class")["source"]
    .agg(["nunique", lambda s: ", ".join(sorted(set(s)))])
    .rename(columns={"nunique": "n_sources", "<lambda_0>": "sources"})
)
multi = overlap[overlap["n_sources"] > 1].sort_values("n_sources", ascending=False)
print(f"{len(multi)} classes appear in more than one source (dedupe/merge needed):")
multi

## 4. Image quality — resolution sample & corrupt-file check

In [ ]:
import random

random.seed(42)
sample_stats, corrupt = [], []
for src in sources:
    files = [f for f in Path(src).rglob("*") if f.suffix.lower() in IMG_EXTS]
    for f in random.sample(files, min(300, len(files))):
        try:
            with Image.open(f) as im:
                w, h = im.size
            sample_stats.append(
                {"source": src.name, "width": w, "height": h, "min_side": min(w, h)}
            )
        except Exception:
            corrupt.append(str(f))

q = pd.DataFrame(sample_stats)
print(f"corrupt files in sample: {len(corrupt)}")
q.groupby("source")["min_side"].describe()[["count", "min", "25%", "50%", "max"]]

## 5. Coverage of the draft label set

In [ ]:
# Manual aliases: dataset folder name -> our label (extend as the audit reveals more)
ALIASES = {
    "chapati": "roti",
    "chapathi": "roti",
    "vada": "medu_vada",
    "chai": "masala_chai",
    "biryani": "chicken_biryani",
    "carrot_halwa": "gajar_halwa",
    "bell_pepper": "capsicum",
    "litchi": "litchi",
    "lychee": "litchi",
}

df["label"] = df["class"].map(lambda c: ALIASES.get(c, c))
avail = df[df["label"].isin(draft_labels)].groupby("label")["n_images"].sum()

covered = avail[avail >= 150]
thin = avail[avail < 150]
missing = sorted(draft_labels - set(avail.index))

print(f"✅ covered (≥150 imgs): {len(covered)}")
print(f"⚠️ thin (<150 imgs):   {len(thin)} → {sorted(thin.index)}")
print(f"❌ missing entirely:    {len(missing)} → {missing}")

## 6. Decisions

*(fill in after running on the real downloads)*

- Final v1 label set: keep classes with ≥150 images; drop or merge the rest.
- Sources to use / drop, and why.
- Aliases confirmed for the merge step (`prepare_data.py`, Day 5).
- Known quality issues to handle (min resolution filter, corrupt files, near-duplicates).